In [2]:
import numpy as np

np.random.seed(42)

m = 100

X = np.random.randn(4, m)

Y = (np.random.rand(1, m) > 0.5).astype(int)

print(X.shape)
print(Y.shape)

(4, 100)
(1, 100)


In [3]:
# GRADED FUNCTION: forward_propagation

def forward_propagation(x, theta):
    J = theta * x
    return J

In [4]:
x, theta = 2, 4
J = forward_propagation(x, theta)
print ("J = " + str(J))

J = 8


In [5]:
# GRADED FUNCTION: backward_propagation

def backward_propagation(x, theta):
    dtheta = x
    return dtheta

In [6]:
x, theta = 3, 4
dtheta = backward_propagation(x, theta)
print ("dtheta = " + str(dtheta))

dtheta = 3


In [7]:
# GRADED FUNCTION: gradient_check

def gradient_check(x, theta, epsilon=1e-7, print_msg=False):
    theta_plus = theta + epsilon
    theta_minus = theta - epsilon
    J_plus = forward_propagation(x, theta_plus)
    J_minus = forward_propagation(x, theta_minus)
    gradapprox = (J_plus - J_minus) / (2 * epsilon)
    grad = backward_propagation(x, theta)
    numerator = np.linalg.norm(grad - gradapprox)
    denominator = np.linalg.norm(grad) + np.linalg.norm(gradapprox)
    difference = numerator / denominator
    if print_msg:
        if difference > 2e-7:
            print ("\033[93m" + "There is a mistake in the backward propagation! difference = " + str(difference) + "\033[0m")
        else:
            print ("\033[92m" + "Your backward propagation works perfectly fine! difference = " + str(difference) + "\033[0m")
    
    return difference

In [8]:
x, theta = 3, 4
difference = gradient_check(x, theta, print_msg=True)

Your backward propagation works perfectly fine! difference = 7.814075313343006e-11


In [9]:
def forward_propagation_n(X, Y, parameters):
    
    # retrieve parameters
    m = X.shape[1]
    W1 = parameters["W1"]
    b1 = parameters["b1"]
    W2 = parameters["W2"]
    b2 = parameters["b2"]
    W3 = parameters["W3"]
    b3 = parameters["b3"]

    # LINEAR -> RELU -> LINEAR -> RELU -> LINEAR -> SIGMOID
    Z1 = np.dot(W1, X) + b1
    A1 = relu(Z1)
    Z2 = np.dot(W2, A1) + b2
    A2 = relu(Z2)
    Z3 = np.dot(W3, A2) + b3
    A3 = sigmoid(Z3)

    # Cost
    log_probs = np.multiply(-np.log(A3),Y) + np.multiply(-np.log(1 - A3), 1 - Y)
    cost = 1. / m * np.sum(log_probs)
    
    cache = (Z1, A1, W1, b1, Z2, A2, W2, b2, Z3, A3, W3, b3)
    
    return cost, cache

In [10]:
def backward_propagation_n(X, Y, cache):
    
    m = X.shape[1]
    (Z1, A1, W1, b1, Z2, A2, W2, b2, Z3, A3, W3, b3) = cache
    
    dZ3 = A3 - Y
    dW3 = 1. / m * np.dot(dZ3, A2.T)
    db3 = 1. / m * np.sum(dZ3, axis=1, keepdims=True)
    
    dA2 = np.dot(W3.T, dZ3)
    dZ2 = np.multiply(dA2, np.int64(A2 > 0))
    dW2 = 1. / m * np.dot(dZ2, A1.T) * 2
    db2 = 1. / m * np.sum(dZ2, axis=1, keepdims=True)
    
    dA1 = np.dot(W2.T, dZ2)
    dZ1 = np.multiply(dA1, np.int64(A1 > 0))
    dW1 = 1. / m * np.dot(dZ1, X.T)
    db1 = 4. / m * np.sum(dZ1, axis=1, keepdims=True)
    
    gradients = {"dZ3": dZ3, "dW3": dW3, "db3": db3,
                 "dA2": dA2, "dZ2": dZ2, "dW2": dW2, "db2": db2,
                 "dA1": dA1, "dZ1": dZ1, "dW1": dW1, "db1": db1}
    
    return gradients

In [11]:
# GRADED FUNCTION: gradient_check_n

def gradient_check_n(parameters, gradients, X, Y, epsilon=1e-7, print_msg=False):
 
    # Set-up variables
    parameters_values, _ = dictionary_to_vector(parameters)
    
    grad = gradients_to_vector(gradients)
    num_parameters = parameters_values.shape[0]
    J_plus = np.zeros((num_parameters, 1))
    J_minus = np.zeros((num_parameters, 1))
    gradapprox = np.zeros((num_parameters, 1))
    
    # Compute gradapprox
    for i in range(num_parameters):
 
        theta_plus = np.copy(parameters_values)
        theta_plus[i][0] = theta_plus[i][0] + epsilon
        J_plus[i], _ = forward_propagation_n(
                X,
                Y,
                vector_to_dictionary(theta_plus)
            )
       
        theta_minus = np.copy(parameters_values)
        theta_minus[i][0] = theta_minus[i][0] - epsilon

        J_minus[i], _ = forward_propagation_n(
            X,
            Y,
            vector_to_dictionary(theta_minus)
        )

        
        gradapprox[i] = (J_plus[i] - J_minus[i]) / (2 * epsilon)
  
    numerator = np.linalg.norm(grad - gradapprox)

    denominator = (
        np.linalg.norm(grad)
        + np.linalg.norm(gradapprox)
    )

    difference = numerator / denominator
    
    if print_msg:
        if difference > 2e-7:
            print ("\033[93m" + "There is a mistake in the backward propagation! difference = " + str(difference) + "\033[0m")
        else:
            print ("\033[92m" + "Your backward propagation works perfectly fine! difference = " + str(difference) + "\033[0m")

    return difference

In [17]:
#Activation functions
def relu(Z):
    return np.maximum(0, Z)

def sigmoid(Z):
    return 1 / (1 + np.exp(-Z))

In [15]:
import numpy as np

X = np.random.randn(4, 3)
Y = np.array([[1, 1, 0]])

parameters = {
    "W1": np.random.randn(5, 4),
    "b1": np.random.randn(5, 1),
    "W2": np.random.randn(3, 5),
    "b2": np.random.randn(3, 1),
    "W3": np.random.randn(1, 3),
    "b3": np.random.randn(1, 1)
}

In [19]:
def dictionary_to_vector(parameters):
    

    keys = []
    count = 0

    for key in ["W1","b1","W2","b2","W3","b3"]:

        new_vector = np.reshape(parameters[key], (-1,1))

        keys = keys + [key] * new_vector.shape[0]

        if count == 0:
            theta = new_vector
        else:
            theta = np.concatenate((theta, new_vector), axis=0)

        count += 1

    return theta, keys

In [21]:

def gradients_to_vector(gradients):

    count = 0

    for key in ["dW1","db1","dW2","db2","dW3","db3"]:

        new_vector = np.reshape(gradients[key], (-1,1))

        if count == 0:
            theta = new_vector
        else:
            theta = np.concatenate((theta, new_vector), axis=0)

        count += 1

    return theta

In [23]:
def vector_to_dictionary(theta):
    """
    Unrolls a parameter vector into a parameter dictionary.
    """

    parameters = {}

    parameters["W1"] = theta[:20].reshape((5,4))
    parameters["b1"] = theta[20:25].reshape((5,1))

    parameters["W2"] = theta[25:40].reshape((3,5))
    parameters["b2"] = theta[40:43].reshape((3,1))

    parameters["W3"] = theta[43:46].reshape((1,3))
    parameters["b3"] = theta[46:47].reshape((1,1))

    return parameters

In [25]:
cost, cache = forward_propagation_n(X, Y, parameters)
gradients = backward_propagation_n(X, Y, cache)
difference = gradient_check_n(parameters, gradients, X, Y, 1e-7, True)
expected_values = [0.2850931567761623, 1.1890913024229996e-07]
assert not(type(difference) == np.ndarray), "You are not using np.linalg.norm for numerator or denominator"
assert type(difference) == np.float64, "The value must be a 64 bit floating point scalar."


There is a mistake in the backward propagation! difference = 0.33182189132346745


Conclusion:
- For 1D gradient checking, the difference = 7.814075313343006e-11, which means that the gradients computed by backward propagation are correct.
- For n D gradient checking, the difference = 0.33182189132346745, which means that the gradients computed by backward propagation are not correct. 